# CineMatch: Content-Based Movie Recommendation Engine

This notebook builds a movie-discovery engine from TMDB metadata. It represents each film using genres, keywords, cast, directors, and plot descriptions, then retrieves similar titles with TF-IDF and cosine similarity.

**Goal:** demonstrate a transparent, reusable content-based recommendation workflow. This approach does not require user ratings or watch-history data.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from cinematch.config import CREDITS_FILE, MOVIES_FILE
from cinematch.data import load_tmdb_data
from cinematch.evaluation import genre_overlap_evaluation
from cinematch.recommender import CineMatchRecommender

if not MOVIES_FILE.exists() or not CREDITS_FILE.exists():
    raise FileNotFoundError("Add tmdb_5000_movies.csv and tmdb_5000_credits.csv to the data/ folder before running this notebook.")


## Load and inspect data

The two source files are merged on the TMDB movie identifier. The project keeps raw data local rather than committing dataset files to GitHub.


In [ ]:
movies = load_tmdb_data(str(MOVIES_FILE), str(CREDITS_FILE))
print(f"Merged movies: {movies.shape[0]:,}")
movies[["id", "title", "genres", "keywords", "cast", "crew"]].head(3)


## Build recommendation index

The reusable `CineMatchRecommender` safely parses nested metadata fields, constructs a text profile for each movie, and vectorizes it using TF-IDF.


In [ ]:
recommender = CineMatchRecommender(max_features=20_000, ngram_range=(1, 2))
recommender.fit(movies)

recommender.items[["title", "genres_list", "cast_list", "director", "metadata"]].head(3)


## Retrieve similar movies

Change the input title below to explore the catalog. Similarity scores reflect overlap in the metadata representation; they are not ratings or predicted enjoyment scores.


In [ ]:
query_title = "Avatar"
recommendations = recommender.recommend(query_title, top_k=10)
recommendations


## Offline evaluation

A recommendation is treated as relevant if it shares at least one genre with the query movie. This is a lightweight sanity check, not a substitute for user testing, diversity metrics, or personalized ranking evaluation.


In [ ]:
evaluation = genre_overlap_evaluation(recommender, top_k=10, sample_size=300, random_state=42)
evaluation


## Next steps

- Add a Streamlit interface with title search, poster images, and filters.
- Add plot embeddings to capture semantic similarity beyond keyword overlap.
- Introduce diversity-aware re-ranking to avoid overly repetitive results.
- Evaluate with user feedback, NDCG, novelty, catalog coverage, and diversity.
